In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_identification_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyIdentificationId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "PartyIdentificationKey string,"
        #     "PartyKey string,"
        #     "CDI string,"
        #     "PartyIdentificationTypeCode string,"
        #     "PartyIdentificationType string,"
        #     "PartyIdentificationNumber string,"
        #     "SourceTable string,"
        #     "MainFlag string,"
        #     "IssueAuthority string,"
        #     "IssueDate date,"
        #     "ExpiredDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PartyIdentification"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # logging.info("Delta Table has been created!")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "CUST_ID",
            "ID_NO",
            "CUST_ID",
            "CDI",
            "ISSUE_DATE",
            "unit_code",
            "businessdate",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "portfolioname",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== other_id ==================

        otherid_extra_filters = None

        if otherid_extra_filters is not None:
            otherid_extra_filters.append(date_filter)
            otherid_final_filters = " AND ".join(otherid_extra_filters)
        else:
            otherid_final_filters = date_filter

        otherid_selected_fields = [
            "FK_CUSTOMERCUST_ID",
            "SERIAL_NO",
            "FKGD_HAS_TYPE",
            "FKGD_HAS_BEEN_ISSU",
            "EXPIRY_DATE",
            "TMSTAMP",
            "ISSUE_DATE",
            "MAIN_FLAG",
            "FKGH_HAS_BEEN_ISSU",
            "FKGH_HAS_TYPE",
            "ID_NO",
            "ISSUE_AUTHORITY",
            "INCOMPLETE_U_COMNT",
            "BusinessDate",
        ]

        otherid = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_other_id",
            otherid_final_filters,
            otherid_selected_fields,
        )

        # ================== params ==================

        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)
        else:
            params_final_filters = date_filter

        params_selected_fields = [
            "param_type",
            "serial_num",
            "description_orig",
        ]

        params = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        )

        # ================== party_identification_profits1 ==================

        # ================== Join ==================
        PartyIdentificationId_primary_key_fields1 = [
            customer.CUST_ID,
            customer.ID_NO,
            lit("IdentityCard"),
            lit("profits_customer"),
            lit("profits"),
        ]
        PartyIdentificationKey_primary_key_fields1 = [
            customer.CUST_ID,
            customer.ID_NO,
            lit("IdentityCard"),
            lit("profits_customer"),
        ]

        party_identification_profits1 = customer.join(
            portfolio,
            customer.unit_code == portfolio.unit_code_profits,
            "left",
        )

        party_identification_profits1 = party_identification_profits1.withColumn(
            "PartyIdentificationId",
            sha2(concat_ws(";", *PartyIdentificationId_primary_key_fields1), 256),
        ).withColumn(
            "PartyIdentificationKey",
            concat_ws(";", *PartyIdentificationKey_primary_key_fields1),
        )

        party_identification_profits1 = party_identification_profits1.select(
            party_identification_profits1.PartyIdentificationId.cast("string").alias(
                "PartyIdentificationId"
            ),
            lit("profits").alias("SourceSystem"),
            party_identification_profits1.unit_code.cast("string").alias(
                "PortfolioKey"
            ),
            portfolio.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            party_identification_profits1.PartyIdentificationKey.cast("string").alias(
                "PartyIdentificationKey"
            ),
            party_identification_profits1.CUST_ID.cast("string").alias("PartyKey"),
            party_identification_profits1.CDI.cast("string").alias("CDI"),
            lit("IdentityCard").alias("PartyIdentificationTypeCode"),
            lit("IdentityCard").alias("PartyIdentificationType"),
            party_identification_profits1.ID_NO.cast("string").alias(
                "PartyIdentificationNumber"
            ),
            lit("profits_customer").cast("string").alias("SourceTable"),
            lit(None).cast("string").alias("MainFlag"),
            lit(None).cast("string").alias("IssueAuthority"),
            party_identification_profits1.ISSUE_DATE.cast("date").alias("IssueDate"),
            lit(None).alias("ExpiredDate"),
            party_identification_profits1.businessdate.alias("BusinessDate"),
        )

        # ================== party_identification_profits2 ==================

        # ================== Join ==================
        PartyIdentificationId_primary_key_fields2 = [
            otherid.FK_CUSTOMERCUST_ID,
            otherid.ID_NO,
            params.description_orig,
            lit("profits_other_id"),
            lit("profits"),
        ]
        PartyIdentificationKey_primary_key_fields2 = [
            otherid.FK_CUSTOMERCUST_ID,
            otherid.ID_NO,
            params.description_orig,
            lit("profits_other_id"),
        ]

        party_identification_profits2 = (
            otherid.join(
                params,
                (otherid.FKGH_HAS_TYPE == params.param_type)
                & (otherid.SERIAL_NO == params.serial_num),
                "left",
            )
            .join(
                customer,
                customer.CUST_ID == otherid.FK_CUSTOMERCUST_ID,
                "left",
            )
            .join(
                portfolio,
                customer.unit_code == portfolio.unit_code_profits,
                "left",
            )
        )

        party_identification_profits2 = party_identification_profits2.withColumn(
            "PartyIdentificationId",
            sha2(concat_ws(";", *PartyIdentificationId_primary_key_fields2), 256),
        ).withColumn(
            "PartyIdentificationKey",
            concat_ws(";", *PartyIdentificationKey_primary_key_fields2),
        )

        party_identification_profits2 = party_identification_profits2.select(
            party_identification_profits2.PartyIdentificationId.cast("string").alias(
                "PartyIdentificationId"
            ),
            lit("profits").alias("SourceSystem"),
            party_identification_profits2.unit_code.cast("string").alias(
                "PortfolioKey"
            ),
            portfolio.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            party_identification_profits2.PartyIdentificationKey.cast("string").alias(
                "PartyIdentificationKey"
            ),
            party_identification_profits2.FK_CUSTOMERCUST_ID.cast("string").alias(
                "PartyKey"
            ),
            party_identification_profits2.CDI.cast("string").alias("CDI"),
            party_identification_profits2.SERIAL_NO.alias(
                "PartyIdentificationTypeCode"
            ),
            party_identification_profits2.description_orig.alias(
                "PartyIdentificationType"
            ),
            otherid.ID_NO.cast("string").alias("PartyIdentificationNumber"),
            lit("profits_other_id").cast("string").alias("SourceTable"),
            party_identification_profits2.MAIN_FLAG.cast("string").alias("MainFlag"),
            party_identification_profits2.ISSUE_AUTHORITY.cast("string").alias(
                "IssueAuthority"
            ),
            otherid.ISSUE_DATE.cast("date").alias("IssueDate"),
            party_identification_profits2.EXPIRY_DATE.cast("date").alias("ExpiredDate"),
            otherid.BusinessDate.alias("BusinessDate"),
        )

        # ================== Union ==================

        party_identification_profits = party_identification_profits1.union(
            party_identification_profits2
        )

        # ================== Append ==================

        party_identification_profits.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_identification_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)